# 🎨 AUTOMATIC1111 WebUI (Lobe Theme + Bộ Trợ lý Prompt Toàn diện)

Notebook triển khai **AUTOMATIC1111 WebUI** với giao diện **Lobe Theme** cao cấp & trợ lý gợi ý từ khóa prompt:
- 💎 **Giao diện Lobe Theme**: Hiện đại phong cách Midjourney / Apple, Dark mode, Glassmorphism, tối giản và chuyên nghiệp.
- 🧠 **Trợ lý Prompt All-In-One (`sd-webui-prompt-all-in-one`)**:
  - 🇻🇳 **Dịch trực tiếp tiếng Việt -> tiếng Anh** ngay trong WebUI, không cần mở Google Dịch.
  - 📚 **Từ điển từ khóa phân loại sẵn**: Khuôn mặt, Tóc, Áo quần, Tư thế, Góc chụp, Ánh sáng, Chất lượng... chỉ cần click chọn.
  - 🎴 **Tự động nhận diện LoRA**: Hiển thị danh sách LoRA có sẵn để bấm 1-click chèn vào prompt kèm thanh chỉnh weight.
- ⚡ **Thư viện Style Mẫu (`styles.csv`)**: Tích hợp sẵn hàng loạt style (Krea 2 Succubus, Realistic Vision, SDXL Lightning, Chân dung nghệ thuật...).
- 💾 **Dùng chung kho Model với ComfyStudio trên Google Drive** (`MyDrive/ComfyStudio/models`): Không cần tải lại.
- 🚀 **Download đa luồng `aria2c`**: Bypass lỗi 403 Civitai, tải tốc độ tối đa.


In [ ]:
#@title 1. Kết nối Google Drive & Cấu hình Thư mục Dùng chung
import os
import shutil
from google.colab import drive

print("🚀 Đang kết nối Google Drive...")
drive.mount('/content/drive')

# Thư mục gốc lưu trữ model trên Drive (dùng chung với ComfyStudio)
DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
DRIVE_OUTPUTS = "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111"

# Các thư mục model cần thiết cho Automatic1111
FOLDERS = [
    "checkpoints",
    "loras",
    "vae",
    "controlnet",
    "upscale_models",
    "embeddings"
]

for f in FOLDERS:
    os.makedirs(os.path.join(DRIVE_BASE, f), exist_ok=True)
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

print(f"✅ Đã sẵn sàng thư mục Models trên Drive: {DRIVE_BASE}")
print(f"📁 Thư mục xuất ảnh tự động: {DRIVE_OUTPUTS}")


In [ ]:
#@title 2. Cài đặt AUTOMATIC1111 + Lobe Theme + Bộ Trợ lý Prompt All-In-One
import os
import shutil
import subprocess

WEBUI_DIR = "/content/stable-diffusion-webui"
EXT_DIR = "/content/stable-diffusion-webui/extensions"

# 1. Clone Automatic1111 WebUI
if not os.path.exists(WEBUI_DIR):
    print("📦 Đang clone AUTOMATIC1111 Stable Diffusion WebUI...")
    !git clone --depth 1 https://github.com/AUTOMATIC1111/stable-diffusion-webui.git /content/stable-diffusion-webui
else:
    print("ℹ️ Thư mục WebUI đã tồn tại, cập nhật code mới nhất...")
    %cd /content/stable-diffusion-webui
    !git pull

%cd /content/stable-diffusion-webui

# 2. Cài đặt aria2 và thư viện phụ trợ xformers
print("⚡ Cài đặt aria2 và xformers...")
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q xformers==0.0.28.post1 --index-url https://download.pytorch.org/whl/cu124

# 3. Cài đặt bộ Extensions & Giao diện Lobe Theme
os.makedirs(EXT_DIR, exist_ok=True)

# A. Giao diện Lobe Theme (Hiện đại phong cách Apple / Midjourney)
if not os.path.exists(f"{EXT_DIR}/sd-webui-lobe-theme"):
    print("💎 Đang cài đặt Giao diện Lobe Theme...")
    !git clone --depth 1 https://github.com/canisminor1990/sd-webui-lobe-theme.git {EXT_DIR}/sd-webui-lobe-theme

# B. Trợ lý Prompt All-In-One (Dịch tiếng Việt, Từ điển từ khóa phân loại, Quản lý LoRA)
if not os.path.exists(f"{EXT_DIR}/sd-webui-prompt-all-in-one"):
    print("🧠 Đang cài đặt Trợ lý Prompt All-In-One (Dịch thuật + Từ điển tag)...")
    !git clone --depth 1 https://github.com/Physton/sd-webui-prompt-all-in-one.git {EXT_DIR}/sd-webui-prompt-all-in-one

# C. Tag Autocomplete (Tự động gợi ý từ khóa khi gõ)
if not os.path.exists(f"{EXT_DIR}/a1111-sd-webui-tagcomplete"):
    print("🔌 Đang cài đặt Extension Tag Autocomplete...")
    !git clone --depth 1 https://github.com/DominikDoom/a1111-sd-webui-tagcomplete.git {EXT_DIR}/a1111-sd-webui-tagcomplete

# D. ControlNet Extension
if not os.path.exists(f"{EXT_DIR}/sd-webui-controlnet"):
    print("🔌 Đang cài đặt Extension sd-webui-controlnet...")
    !git clone --depth 1 https://github.com/Mikubill/sd-webui-controlnet.git {EXT_DIR}/sd-webui-controlnet

# 4. Tạo sẵn bộ Style Mẫu (styles.csv) để nạp 1-click trong WebUI
styles_content = """name,prompt,negative_prompt
"🔥 Krea 2 / LUSTIFY Succubus","A woman is captured as a melancholic succubus with horns and wings, posed against a backdrop of ethereal white cherry blossoms and cool blue atmospheric lighting, very realistic, 8k, cinematic photography, <lora:KNP_000003000:1.0>","worst quality, low quality, bad anatomy, deformed, distorted, blurry, cartoon, 3d render"
"📷 Realistic Vision V6 (Paris Candid)","RAW street style photography of a beautiful model in Paris, natural candid smile, stylish beige trench coat, soft golden hour sunlight, 50mm f/1.8, bokeh, hyperdetailed skin, authentic look, 8k","deformed, bad anatomy, disfigured, poorly drawn face, mutation, extra limb, ugly, disgusting, blurred, watermark, bad hands, cartoon, 3d render"
"⚡ SDXL Lightning Master (8K Portrait)","Award-winning RAW cinematic portrait of an elegant woman, authentic skin pores, fine skin texture, soft chiaroscuro studio lighting, 85mm f/1.4 lens, natural gaze, luxury atmosphere, photorealistic, 8k uhd, masterpiece, <lora:add-detail-xl:0.8>","ugly, deformed, noisy, blurry, low contrast, cartoon, anime, 3d render, extra limbs, bad hands, plastic skin"
"💎 Asian Beauty Elegance","RAW close-up portrait of a gorgeous 22yo Asian woman, natural glowing skin texture, subtle makeup, traditional silk details, soft cinematic lighting, 85mm lens, 8k uhd, photorealistic","disfigured, ugly, bad art, deformed, poorly drawn, extra limbs, dull eyes, low quality"
"✨ Detail & Skin Booster","hyperdetailed skin texture, visible pores, individual eyelashes, soft rim lighting, sharp focus, 85mm portrait photography, 8k uhd, professional color grading","plastic skin, airbrushed, wax figure, smooth skin, doll, oversaturated"
""".strip()

with open("/content/stable-diffusion-webui/styles.csv", "w", encoding="utf-8") as f:
    f.write(styles_content)
print("📋 Đã cài sẵn 5 bộ Styles Prompt mẫu vào styles.csv!")

# 5. Tạo Symlink liên kết thư mục Models từ Google Drive vào Automatic1111
DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
LINK_MAP = {
    f"{DRIVE_BASE}/checkpoints": "/content/stable-diffusion-webui/models/Stable-diffusion",
    f"{DRIVE_BASE}/loras": "/content/stable-diffusion-webui/models/Lora",
    f"{DRIVE_BASE}/vae": "/content/stable-diffusion-webui/models/VAE",
    f"{DRIVE_BASE}/controlnet": "/content/stable-diffusion-webui/models/ControlNet",
    f"{DRIVE_BASE}/embeddings": "/content/stable-diffusion-webui/embeddings",
    f"{DRIVE_BASE}/upscale_models": "/content/stable-diffusion-webui/models/ESRGAN"
}

for src, dst in LINK_MAP.items():
    if os.path.exists(src):
        if os.path.exists(dst) and not os.path.islink(dst):
            for item in os.listdir(dst):
                s_path = os.path.join(dst, item)
                d_path = os.path.join(src, item)
                if not os.path.exists(d_path):
                    shutil.move(s_path, d_path)
            shutil.rmtree(dst)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            print(f"🔗 Symlink thành công: {src} -> {dst}")

print("\n✅ Cài đặt hoàn tất! Giao diện Lobe Theme & Trợ lý Prompt đã sẵn sàng.")


In [ ]:
#@title 📦 3. Cài Đặt Danh Sách Model Tuyển Chọn (Checklist & Trọn bộ Full Package)
#@markdown Chọn các model bạn muốn sử dụng cho AUTOMATIC1111. Có thể tick lẻ từng model hoặc tick **Tron_Bo_Full_Package** để chọn tất cả.
#@markdown - 🔍 **Chế độ kiểm tra**: Quét xem model nào đã có trên Google Drive (`MyDrive/ComfyStudio/models`), model nào chưa có mà **không tốn dung lượng tải**.
#@markdown - ⚡ **Bảo vệ Google Drive**: Tự động bỏ qua (Skip) các model đã có sẵn trên Drive. Không bao giờ tải lại gây lãng phí.
#@markdown - 🤝 **Dùng chung với ComfyUI**: Model tải ở đây hay ở ComfyStudio đều được lưu chung vào 1 nơi trên Google Drive, không bị nhân bản file!

# --- CÁC TÙY CHỌN CHÍNH ---
Che_Do = "🔍 1. Kiểm tra trạng thái trên Google Drive (Xem cái nào đã có / chưa có)" #@param ["🔍 1. Kiểm tra trạng thái trên Google Drive (Xem cái nào đã có / chưa có)", "⬇️ 2. Tải các model đã tick chọn (Tự động bỏ qua file đã có)"]
Tron_Bo_Full_Package = False #@param {type:"boolean"}

# --- DANH SÁCH CHECKBOX CÁC MODEL ---
# Checkpoint SDXL & SD1.5
Krea_2_Lustify_Turbo_SDXL = True #@param {type:"boolean"}
Realistic_Vision_V6_SD15 = True #@param {type:"boolean"}
SDXL_Lightning_4Step = True #@param {type:"boolean"}
SDXL_Turbo_1Step = False #@param {type:"boolean"}
RealVisXL_V4_SDXL = False #@param {type:"boolean"}

# LoRAs (Trong A1111: gõ <lora:tên_file_không_đuôi:1.0> trong prompt)
Lora_Krea_2_NSFW_V4 = True #@param {type:"boolean"}
Lora_Add_Detail_XL = True #@param {type:"boolean"}

# VAE, Upscaler & ControlNet
VAE_SDXL_Fix = True #@param {type:"boolean"}
Upscaler_4x_UltraSharp = True #@param {type:"boolean"}
ControlNet_OpenPose = False #@param {type:"boolean"}

# --- MẬT KHẨU MÃ HÓA & API TOKENS ---
#@markdown 🔐 **Bảo mật**: Nhập mật khẩu để tự động mở khóa 3 Token (Civitai, HuggingFace, Ngrok). Nếu tự điền ô bên dưới thì sẽ ưu tiên dùng ô đó.
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}
Civitai_API_Token = "" #@param {type:"string"}
HuggingFace_Token = "" #@param {type:"string"}

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open("/content/.vault_tokens.json", "w") as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = "", pwd: str = "") -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists("/content/.vault_tokens.json"):
            cached = json.load(open("/content/.vault_tokens.json"))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ""

# Giải mã token tự động nếu có mật khẩu
Civitai_API_Token = get_vault_token("civitai", Civitai_API_Token, Mat_Khau_Mo_Khoa)
HuggingFace_Token = get_vault_token("huggingface", HuggingFace_Token, Mat_Khau_Mo_Khoa)
if Civitai_API_Token or HuggingFace_Token:
    print("🔓 [Bảo Mật] Đã giải mã thành công Token từ Mật khẩu!")


import os
import sys
import subprocess
from huggingface_hub import hf_hub_download

!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

DRIVE_DIR = "/content/drive/MyDrive/ComfyStudio/models"
A1111_DIR = "/content/stable-diffusion-webui/models"
has_drive = os.path.exists("/content/drive/MyDrive")
TARGET_BASE = DRIVE_DIR if has_drive else A1111_DIR

CURATED_MODELS = [
    {
        "id": "krea2_turbo",
        "selected": Krea_2_Lustify_Turbo_SDXL,
        "name": "Krea 2 Turbo (LUSTIFY!) [SDXL 4-Step]",
        "category": "checkpoints",
        "filename": "lustify_workflows_krea_2_sdxl.safetensors",
        "min_size_mb": 5000,
        "source_type": "civitai",
        "model_version_id": "3112728",
        "url": "https://civitai.com/api/download/models/3112728",
        "desc": "SDXL Turbo 4-step cực nét (render 2-4 giây)"
    },
    {
        "id": "realistic_vision_v6",
        "selected": Realistic_Vision_V6_SD15,
        "name": "Realistic Vision V6.0 B1 [SD1.5]",
        "category": "checkpoints",
        "filename": "realisticVisionV60B1_v51VAE.safetensors",
        "min_size_mb": 1800,
        "source_type": "civitai",
        "model_version_id": "130072",
        "url": "https://civitai.com/api/download/models/130072?type=Model&format=SafeTensor&size=pruned&fp=fp16",
        "alt_hf": {"repo_id": "SG161222/Realistic_Vision_V5.1_noVAE", "filename": "Realistic_Vision_V5.1_fp16-no-ema.safetensors"},
        "desc": "SD1.5 người thật photorealistic cực chuẩn"
    },
    {
        "id": "sdxl_lightning",
        "selected": SDXL_Lightning_4Step,
        "name": "SDXL Lightning 4-Step (ByteDance)",
        "category": "checkpoints",
        "filename": "sdxl_lightning_4step.safetensors",
        "min_size_mb": 5000,
        "source_type": "hf",
        "repo_id": "ByteDance/SDXL-Lightning",
        "hf_filename": "sdxl_lightning_4step.safetensors",
        "desc": "SDXL tốc độ cao 2 giây/ảnh"
    },
    {
        "id": "sdxl_turbo",
        "selected": SDXL_Turbo_1Step,
        "name": "SDXL Turbo FP16 (Stability AI)",
        "category": "checkpoints",
        "filename": "sd_xl_turbo_1.0_fp16.safetensors",
        "min_size_mb": 3000,
        "source_type": "hf",
        "repo_id": "stabilityai/sdxl-turbo",
        "hf_filename": "sd_xl_turbo_1.0_fp16.safetensors",
        "desc": "SDXL siêu tốc 1 bước render"
    },
    {
        "id": "realvisxl_v4",
        "selected": RealVisXL_V4_SDXL,
        "name": "RealVisXL V4.0 (SDXL Chân Thực)",
        "category": "checkpoints",
        "filename": "realvisxlV40_v40Bakedvae.safetensors",
        "min_size_mb": 5000,
        "source_type": "hf",
        "repo_id": "SG161222/RealVisXL_V4.0",
        "hf_filename": "RealVisXL_V4.0.safetensors",
        "desc": "SDXL chân thực cao cấp 15-20s"
    },
    {
        "id": "lora_krea_2_nsfw",
        "selected": Lora_Krea_2_NSFW_V4,
        "name": "LoRA Krea 2 NSFW V4",
        "category": "loras",
        "filename": "krea_2_nsfw_v4.safetensors",
        "min_size_mb": 150,
        "source_type": "civitai",
        "model_version_id": "3147117",
        "url": "https://civitai.com/api/download/models/3147117",
        "desc": "LoRA chuyên dụng cho Krea 2 Turbo"
    },
    {
        "id": "lora_add_detail",
        "selected": Lora_Add_Detail_XL,
        "name": "LoRA Add-Detail-XL",
        "category": "loras",
        "filename": "add-detail-xl.safetensors",
        "min_size_mb": 30,
        "source_type": "hf",
        "repo_id": "OedoSoldier/detail-tweaker-lora",
        "hf_filename": "add-detail-xl.safetensors",
        "desc": "Tăng độ chi tiết da, mắt, tóc cho SDXL"
    },
    {
        "id": "vae_sdxl",
        "selected": VAE_SDXL_Fix,
        "name": "SDXL VAE (Fix Màu & Khử Mờ)",
        "category": "vae",
        "filename": "sdxl_vae.safetensors",
        "min_size_mb": 200,
        "source_type": "hf",
        "repo_id": "stabilityai/sdxl-vae",
        "hf_filename": "sdxl_vae.safetensors",
        "desc": "VAE chuẩn SDXL"
    },
    {
        "id": "upscaler_4x_ultrasharp",
        "selected": Upscaler_4x_UltraSharp,
        "name": "Upscaler 4x-UltraSharp",
        "category": "upscale_models",
        "filename": "4x-UltraSharp.pth",
        "min_size_mb": 50,
        "source_type": "hf",
        "repo_id": "lokcx/4x-Ultrasharp",
        "hf_filename": "4x-UltraSharp.pth",
        "desc": "Nâng nét ảnh lên 4K"
    },
    {
        "id": "controlnet_openpose",
        "selected": ControlNet_OpenPose,
        "name": "ControlNet OpenPose (SD1.5)",
        "category": "controlnet",
        "filename": "control_v11p_sd15_openpose.pth",
        "min_size_mb": 1000,
        "source_type": "hf",
        "repo_id": "lllyasviel/ControlNet-v1-1",
        "hf_filename": "control_v11p_sd15_openpose.pth",
        "desc": "Khống chế dáng người mẫu"
    }
]

# ==========================================================================
# CHẾ ĐỘ 1: KIỂM TRA TRẠNG THÁI TRÊN GOOGLE DRIVE (KHÔNG TẢI GÌ)
# ==========================================================================
if Che_Do.startswith("🔍"):
    print("=" * 85)
    print("📊 BẢNG KIỂM TRA TRẠNG THÁI MODEL TRÊN GOOGLE DRIVE (CHO AUTOMATIC1111)")
    print(f"📂 Thư mục kiểm tra: {TARGET_BASE}")
    print("=" * 85)
    print(f"{'STT':<4} | {'TRẠNG THÁI':<12} | {'DANH MỤC':<15} | {'DUNG LƯỢNG':<10} | {'TÊN MODEL & FILE'}")
    print("-" * 85)
    
    count_exist = 0
    total_exist_bytes = 0
    
    for idx, item in enumerate(CURATED_MODELS, 1):
        drive_file = os.path.join(TARGET_BASE, item["category"], item["filename"])
        min_bytes = item["min_size_mb"] * 1024 * 1024
        
        is_exist = os.path.exists(drive_file) and os.path.getsize(drive_file) >= min_bytes
        if is_exist:
            count_exist += 1
            file_bytes = os.path.getsize(drive_file)
            total_exist_bytes += file_bytes
            size_str = f"{file_bytes / (1024**3):.2f} GB" if file_bytes >= 1024**3 else f"{file_bytes / (1024**2):.1f} MB"
            status_str = "✅ ĐÃ CÓ"
        else:
            size_str = f"~{item['min_size_mb']} MB"
            status_str = "❌ CHƯA CÓ"
            
        print(f"{idx:<4} | {status_str:<12} | {item['category']:<15} | {size_str:<10} | {item['name']}")
        print(f"     ↳ File: {item['filename']} - ({item['desc']})")
    
    total_exist_gb = total_exist_bytes / (1024**3)
    print("=" * 85)
    print(f"📈 TỔNG KẾT: Đã có {count_exist}/{len(CURATED_MODELS)} model (Dung lượng: {total_exist_gb:.2f} GB). Còn thiếu: {len(CURATED_MODELS) - count_exist} model.")
    print("💡 ĐỂ TẢI CÁC MODEL CÒN THIẾU:")
    print("   1. Tick chọn model cần tải (hoặc bật [Tron_Bo_Full_Package] để tải hết).")
    print("   2. Đổi [Che_Do] sang '⬇️ 2. Tải các model đã tick chọn' rồi bấm Play (▶️)!")
    print("=" * 85)

# ==========================================================================
# CHẾ ĐỘ 2: BẮT ĐẦU TẢI (TỰ ĐỘNG BỎ QUA FILE ĐÃ CÓ)
# ==========================================================================
else:
    print("=" * 85)
    print("🚀 BẮT ĐẦU TIẾN TRÌNH TẢI MODEL VÀO GOOGLE DRIVE")
    print(f"🎯 Thư mục lưu trữ: {TARGET_BASE}")
    print("=" * 85)
    
    targets = [m for m in CURATED_MODELS if Tron_Bo_Full_Package or m["selected"]]
    if not targets:
        print("⚠️ Bạn chưa tick chọn model nào! Hãy tick chọn model hoặc bật [Tron_Bo_Full_Package].")
    
    for item in targets:
        category = item["category"]
        filename = item["filename"]
        min_bytes = item["min_size_mb"] * 1024 * 1024
        dest_dir = os.path.join(TARGET_BASE, category)
        os.makedirs(dest_dir, exist_ok=True)
        dest_file = os.path.join(dest_dir, filename)
        
        # 1. Kiểm tra tồn tại
        if os.path.exists(dest_file) and os.path.getsize(dest_file) >= min_bytes:
            sz = os.path.getsize(dest_file) / (1024**3)
            print(f"⏩ [BỎ QUA - ĐÃ CÓ SẴN] {item['name']} ({sz:.2f} GB) trên Drive -> Không tải lại.")
            continue
        
        print(f"\n⬇️ [ĐANG TẢI] {item['name']}...")
        print(f"   Lưu vào: {dest_file}")
        
        # 2. Tải tùy theo nguồn (Civitai hoặc HuggingFace)
        if item["source_type"] == "civitai":
            url = item["url"]
            if Civitai_API_Token and "token=" not in url:
                url += f"?token={Civitai_API_Token}" if "?" not in url else f"&token={Civitai_API_Token}"
            
            cmd = [
                "aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M",
                "--file-allocation=none",
                "--header=User-Agent: Mozilla/5.0",
                "-d", dest_dir,
                "-o", filename,
                url
            ]
            ret = subprocess.run(cmd)
            if ret.returncode != 0:
                print(f"⚠️ Tải Civitai lỗi. Thử nguồn dự phòng nếu có...")
                if "alt_hf" in item:
                    hf = item["alt_hf"]
                    try:
                        hf_hub_download(repo_id=hf["repo_id"], filename=hf["filename"], local_dir=dest_dir, local_dir_use_symlinks=False)
                        tmp = os.path.join(dest_dir, hf["filename"])
                        if tmp != dest_file and os.path.exists(tmp):
                            os.rename(tmp, dest_file)
                    except Exception as err:
                        print(f"❌ Nguồn HF dự phòng cũng lỗi: {err}")
        
        elif item["source_type"] == "hf":
            try:
                dl_kwargs = {
                    "repo_id": item["repo_id"],
                    "filename": item["hf_filename"],
                    "local_dir": dest_dir,
                    "local_dir_use_symlinks": False
                }
                if HuggingFace_Token:
                    dl_kwargs["token"] = HuggingFace_Token
                dl_path = hf_hub_download(**dl_kwargs)
                if dl_path != dest_file and os.path.exists(dl_path):
                    if os.path.exists(dest_file):
                        os.remove(dest_file)
                    os.rename(dl_path, dest_file)
            except Exception as e:
                print(f"❌ Lỗi tải HuggingFace {filename}: {e}")
        
        # 3. Verify dung lượng sau tải
        if os.path.exists(dest_file) and os.path.getsize(dest_file) >= min_bytes:
            sz = os.path.getsize(dest_file) / (1024**3)
            print(f"✅ Tải thành công: {filename} ({sz:.2f} GB)")
        else:
            print(f"⚠️ Cảnh báo: File {filename} tải về không đủ dung lượng tối thiểu!")
            
    print("\n🎉 Hoàn tất kiểm tra & tải các model đã chọn cho AUTOMATIC1111!")


In [ ]:
#@title 📥 4. Tải Thêm Model Tùy Chọn (Check Trùng Trước Khi Tải)
#@markdown Tải thêm bất kỳ Model nào từ **Civitai** (civitai.com, civitai.red), **HuggingFace**, hoặc Direct Link:
#@markdown - 🔍 **Kiểm tra trước khi tải**: Biết ngay model đã có trong Google Drive chưa, tránh tải trùng.
#@markdown - 🛑 **Tự động Chặn Tải Trùng**: Nếu file đã có sẵn trên Drive, hệ thống sẽ từ chối tải lại để bảo vệ dung lượng.
#@markdown - ⚡ **Bypass 403 Forbidden Civitai**: Tự động nhận diện `modelVersionId` và tải qua API token.
#@markdown - 🤝 **Tự động liên kết**: Tự động nhận diện ngay vào AUTOMATIC1111 mà không cần copy thủ công.

# --- CẤU HÌNH TẢI MODEL ---
Thu_Muc_Luu = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "embeddings"]
Link_Model = "" #@param {type:"string"}
Doi_Ten_File_Tuy_Chon = "" #@param {type:"string"}

# --- THAO TÁC THỰC HIỆN ---
Thao_Tac = "🔍 1. Kiểm tra xem model này đã có trên Google Drive chưa" #@param ["🔍 1. Kiểm tra xem model này đã có trên Google Drive chưa", "⬇️ 2. Bắt đầu tải (Tự động CHẶN nếu đã có sẵn)"]

# --- MẬT KHẨU MÃ HÓA & API TOKENS ---
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}
Civitai_API_Token = "" #@param {type:"string"}
HuggingFace_Token = "" #@param {type:"string"}

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open("/content/.vault_tokens.json", "w") as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = "", pwd: str = "") -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists("/content/.vault_tokens.json"):
            cached = json.load(open("/content/.vault_tokens.json"))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ""

Civitai_API_Token = get_vault_token("civitai", Civitai_API_Token, Mat_Khau_Mo_Khoa)
HuggingFace_Token = get_vault_token("huggingface", HuggingFace_Token, Mat_Khau_Mo_Khoa)


import os
import re
import sys
import json
import urllib.request
import subprocess

!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

DRIVE_DIR = "/content/drive/MyDrive/ComfyStudio/models"
A1111_DIR = "/content/stable-diffusion-webui/models"
has_drive = os.path.exists("/content/drive/MyDrive")
TARGET_BASE = DRIVE_DIR if has_drive else A1111_DIR

def resolve_model_info(url, token=""):
    headers = {"User-Agent": "Mozilla/5.0"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    
    if "civitai.com" in url or "civitai.red" in url:
        ver_match = re.search(r'modelVersionId=(\d+)', url)
        if ver_match:
            ver_id = ver_match.group(1)
            api_url = f"https://civitai.com/api/v1/model-versions/{ver_id}"
            try:
                req = urllib.request.Request(api_url, headers=headers)
                with urllib.request.urlopen(req) as resp:
                    data = json.loads(resp.read().decode('utf-8'))
                    files = data.get("files", [])
                    fname = files[0].get("name") if files else f"{ver_id}.safetensors"
                    size_kb = files[0].get("sizeKB", 0) if files else 0
                    dl_url = f"https://civitai.com/api/download/models/{ver_id}"
                    if token:
                        dl_url += f"?token={token}"
                    return fname, dl_url, size_kb / 1024
            except Exception as e:
                print(f"⚠️ Không đọc được API Civitai: {e}")
                return f"{ver_id}.safetensors", f"https://civitai.com/api/download/models/{ver_id}?token={token}", 0
        
        model_match = re.search(r'/models/(\d+)', url)
        if model_match:
            model_id = model_match.group(1)
            api_url = f"https://civitai.com/api/v1/models/{model_id}"
            try:
                req = urllib.request.Request(api_url, headers=headers)
                with urllib.request.urlopen(req) as resp:
                    data = json.loads(resp.read().decode('utf-8'))
                    versions = data.get("modelVersions", [])
                    if versions:
                        ver = versions[0]
                        ver_id = ver.get("id")
                        files = ver.get("files", [])
                        fname = files[0].get("name") if files else f"{ver_id}.safetensors"
                        size_kb = files[0].get("sizeKB", 0) if files else 0
                        dl_url = f"https://civitai.com/api/download/models/{ver_id}"
                        if token:
                            dl_url += f"?token={token}"
                        return fname, dl_url, size_kb / 1024
            except Exception as e:
                print(f"⚠️ Không đọc được API Civitai: {e}")
                return f"{model_id}.safetensors", url, 0
                
    clean_url = url.split("?")[0]
    fname = os.path.basename(clean_url)
    return fname, url, 0

clean_link = Link_Model.strip()
if not clean_link:
    print("⚠️ VUI LÒNG DÁN LINK MODEL VÀO Ô [Link_Model] Ở TRÊN ĐỂ TIẾP TỤC.")
else:
    detected_name, download_url, estimated_mb = resolve_model_info(clean_link, Civitai_API_Token)
    final_filename = Doi_Ten_File_Tuy_Chon.strip() if Doi_Ten_File_Tuy_Chon.strip() else detected_name
    if not final_filename:
        final_filename = "downloaded_model.safetensors"
        
    target_dir = os.path.join(TARGET_BASE, Thu_Muc_Luu)
    os.makedirs(target_dir, exist_ok=True)
    dest_path = os.path.join(target_dir, final_filename)
    
    is_already_exist = os.path.exists(dest_path) and os.path.getsize(dest_path) > 10 * 1024 * 1024
    
    # ----------------------------------------------------------------------
    # THAO TÁC 1: KIỂM TRA TRƯỚC KHI TẢI
    # ----------------------------------------------------------------------
    if Thao_Tac.startswith("🔍"):
        print("=" * 75)
        print("🔍 KẾT QUẢ KIỂM TRA MODEL TRÊN GOOGLE DRIVE (CHO AUTOMATIC1111)")
        print("=" * 75)
        print(f"📂 Danh mục lưu:   {Thu_Muc_Luu}/")
        print(f"📄 Tên file model: {final_filename}")
        print(f"📍 Đường dẫn đích: {dest_path}")
        print("-" * 75)
        
        if is_already_exist:
            sz_bytes = os.path.getsize(dest_path)
            sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
            print(f"📊 Trạng thái:     ✅ ĐÃ CÓ TRÊN GOOGLE DRIVE! ({sz_str})")
            print("=" * 75)
            print("🛑 KẾT LUẬN: BẠN KHÔNG CẦN TẢI LẠI! Model này đã có sẵn đầy đủ.")
            print("   Hệ thống sẽ bảo vệ Google Drive của bạn khỏi bị tải trùng lãng phí.")
            print("   AUTOMATIC1111 đã tự động nhận diện model này thông qua thư mục dùng chung.")
        else:
            est_str = f"~{estimated_mb:.1f} MB" if estimated_mb > 0 else "Chưa rõ"
            print(f"📊 Trạng thái:     ❌ CHƯA CÓ TRÊN GOOGLE DRIVE (Dung lượng dự kiến: {est_str})")
            print("=" * 75)
            print("💡 BẠN CÓ THỂ TẢI MODEL NÀY:")
            print("   1. Đổi mục [Thao_Tac] ở trên sang: '⬇️ 2. Bắt đầu tải (Tự động CHẶN nếu đã có sẵn)'")
            print("   2. Bấm nút Play (▶️) để bắt đầu tải đa luồng tốc độ cao bằng aria2c!")
            print("=" * 75)
            
    # ----------------------------------------------------------------------
    # THAO TÁC 2: BẮT ĐẦU TẢI
    # ----------------------------------------------------------------------
    else:
        if is_already_exist:
            sz_bytes = os.path.getsize(dest_path)
            sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
            print("=" * 75)
            print(f"🛑 CHẶN TẢI TRÙNG LẶP: Model '{final_filename}' ({sz_str}) ĐÃ CÓ TRÊN GOOGLE DRIVE!")
            print("   Hủy lệnh tải để không tốn thời gian và dung lượng lưu trữ.")
            print("=" * 75)
        else:
            print("=" * 75)
            print(f"⬇️ BẮT ĐẦU TẢI: {final_filename}")
            print(f"   URL: {clean_link}")
            print(f"   Lưu vào: {dest_path}")
            print("=" * 75)
            
            cmd = [
                "aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M",
                "--file-allocation=none",
                "--header=User-Agent: Mozilla/5.0",
                "-d", target_dir,
                "-o", final_filename,
                download_url
            ]
            subprocess.run(cmd)
            
            if os.path.exists(dest_path) and os.path.getsize(dest_path) > 10 * 1024 * 1024:
                sz_bytes = os.path.getsize(dest_path)
                sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
                print("=" * 75)
                print(f"🎉 TẢI THÀNH CÔNG: {final_filename} ({sz_str})")
                print("   Đã sẵn sàng sử dụng trong AUTOMATIC1111!")
                print("=" * 75)
            else:
                print(f"❌ Tải thất bại hoặc file không hợp lệ: {dest_path}")


In [ ]:
#@title 🚀 4. Khởi chạy AUTOMATIC1111 WebUI (Mở Link Truy cập)
#@markdown Chọn phương thức kết nối để lấy URL giao diện WebUI:

tunnel_type = "Gradio Share (Ch\u00ednh ch\u1ee7, d\u1ec5 d\u00f9ng nh\u1ea5t)" #@param ["Gradio Share (Ch\u00ednh ch\u1ee7, d\u1ec5 d\u00f9ng nh\u1ea5t)", "Ngrok Tunnel (Nhanh & \u1ed5n \u0111\u1ecbnh, c\u1ea7n Token)", "LocalTunnel (Mi\u1ec5n ph\u00ed)"]
ngrok_auth_token = "" #@param {type:"string"}
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open("/content/.vault_tokens.json", "w") as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = "", pwd: str = "") -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists("/content/.vault_tokens.json"):
            cached = json.load(open("/content/.vault_tokens.json"))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ""

ngrok_auth_token = get_vault_token("ngrok", ngrok_auth_token, Mat_Khau_Mo_Khoa)
if ngrok_auth_token:
    print("🔓 [Bảo Mật] Đã tự động điền Ngrok Auth Token từ Mật khẩu giải mã!")

theme = "dark" #@param ["dark", "light"]

%cd /content/stable-diffusion-webui

launch_args = [
    "--theme", theme,
    "--enable-insecure-extension-access",
    "--xformers",
    "--no-half-vae",
    "--gradio-queue",
    "--outdir-txt2img-samples", "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111/txt2img",
    "--outdir-img2img-samples", "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111/img2img"
]

if "gradio" in tunnel_type.lower():
    launch_args.append("--share")
elif "ngrok" in tunnel_type.lower():
    if not ngrok_auth_token:
        print("⚠️ Bạn chưa nhập ngrok_auth_token! Đang tự động chuyển sang --share...")
        launch_args.append("--share")
    else:
        launch_args.extend(["--ngrok", ngrok_auth_token])

arg_string = " ".join(launch_args)
print(f"🚀 Đang khởi động WebUI với cờ: {arg_string}")
print("💡 Tìm dòng 'Running on public URL: https://...gradio.live' để nhấn vào mở giao diện WebUI!")

!python launch.py {arg_string}
